# 04 - Idiot Sandwich: TRL DPOTrainer + LoRA persona tuning (Task 5)

Model `HuggingFaceTB/SmolLM2-360M-Instruct`, data `data/persona_lora/ramsay_style_preferences.json` (200 train / 20 val). Accelerator: GPU T4 (one is enough). ~10-20 min.

In [ ]:
REPO_URL = "https://github.com/IsrarAliKhan/small-model-alignment-starter.git"  # edit if your fork lives elsewhere
%cd /kaggle/working
!git clone -q {REPO_URL} || git -C small-model-alignment-starter pull -q
%cd /kaggle/working/small-model-alignment-starter
!git log --oneline -1

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors matplotlib pytest "peft>=0.11.0" "trl==0.24.0"
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("CUDA devices:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
!python scripts/train_lora_dpo.py --output_dir outputs/lora_persona

In [ ]:
!python scripts/plot_metrics.py --runs outputs/lora_persona --keys loss reward_accuracy reward_margin --output outputs/plots/lora_persona.png
from IPython.display import Image; Image('outputs/plots/lora_persona.png')

Sanity check: base vs LoRA on a few held-out persona prompts.

In [ ]:
!python scripts/generate.py --model HuggingFaceTB/SmolLM2-360M-Instruct --chat_template --eval_set eval_sets/persona_style_eval_40.json --output outputs/sanity/persona_base.json --generator smollm2-360m-instruct --limit 4 --max_new_tokens 200
!python scripts/generate.py --model HuggingFaceTB/SmolLM2-360M-Instruct --adapter outputs/lora_persona/adapter --chat_template --eval_set eval_sets/persona_style_eval_40.json --output outputs/sanity/persona_lora.json --generator smollm2-360m-lora-dpo --limit 4 --max_new_tokens 200
import json
for b, l in zip(json.load(open('outputs/sanity/persona_base.json')), json.load(open('outputs/sanity/persona_lora.json'))):
    print('Q:', b['instruction'], '\nBASE:', b['output'][:300], '\nLORA:', l['output'][:300], '\n')

In [ ]:
# Small zip of everything except checkpoints (metrics, configs, plots, generations).
!cd outputs && zip -qr ../lora_persona_run_outputs.zip . -x "*/best/*" "*/final/*" "*/adapter/*" "*/trainer/*" && ls -la ../lora_persona_run_outputs.zip
print("Now click 'Save Version' (Save & Run All or Quick Save) so outputs persist and can be attached later.")